# Tutorial 10: Model Fitting and Parameter Estimation

Fitting ACT-R models to human data is how we test whether our cognitive theories actually match reality. This tutorial covers the main approaches: grid search, optimization algorithms, and Bayesian estimation. We'll also look at how to validate fits and compare competing models.

## 1. Introduction to Model Fitting

The goal is to adjust ACT-R parameters until model predictions match human performance. For example, you might have empirical response time data from a visual search task, and you want to find the retrieval threshold and latency parameters that best reproduce those RTs.

In [ ]:
import pyactr as actr
import numpy as np
import matplotlib.pyplot as plt
from scipy.optimize import minimize
import pandas as pd

# Example human data: response times for different set sizes
human_data = {
    'set_size': [2, 4, 6, 8],
    'mean_rt': [0.45, 0.68, 0.92, 1.15],
    'std_rt': [0.08, 0.10, 0.12, 0.15]
}

def create_search_model(retrieval_threshold, latency_factor, visual_finst_span):
    """Create model with specific parameters"""
    model = actr.ACTRModel()
    model.model_parameters['subsymbolic'] = True
    model.model_parameters['retrieval_threshold'] = retrieval_threshold
    model.model_parameters['latency_factor'] = latency_factor
    model.model_parameters['visual_finst_span'] = visual_finst_span
    
    actr.chunktype("search_goal", "state target_found items_checked")
    actr.chunktype("visual_item", "item_type location")
    
    return model

print("Model fitting framework ready!")
print(f"Human data: {human_data}")

## 2. Grid Search Parameter Fitting

The simplest approach is grid search: try many parameter combinations and pick the best one. It's exhaustive but slow for high-dimensional parameter spaces.

In [ ]:
import pyactr as actr
import numpy as np
from itertools import product

param_grid = {
    'retrieval_threshold': [-1.0, -0.5, 0.0, 0.5],
    'latency_factor': [0.1, 0.2, 0.3, 0.4],
    'latency_exponent': [0.5, 1.0, 1.5]
}

def simulate_response_time(params, set_size):
    """Simulate RT for a given set size"""
    base_time = 0.2
    retrieval_time = params['latency_factor'] * (set_size ** params['latency_exponent'])
    noise = np.random.normal(0, 0.05)
    return base_time + retrieval_time + noise

def compute_rmse(params, human_data):
    """Compute root mean squared error"""
    predicted_rts = []
    for size in human_data['set_size']:
        rts = [simulate_response_time(params, size) for _ in range(10)]
        predicted_rts.append(np.mean(rts))
    
    rmse = np.sqrt(np.mean((np.array(predicted_rts) - 
                           np.array(human_data['mean_rt']))**2))
    return rmse

# Grid search
best_params = None
best_rmse = float('inf')
results = []

param_combinations = list(product(*param_grid.values()))
param_names = list(param_grid.keys())

print(f"Testing {len(param_combinations)} parameter combinations...")

for combo in param_combinations[:10]:
    params = dict(zip(param_names, combo))
    rmse = compute_rmse(params, human_data)
    results.append((params, rmse))
    
    if rmse < best_rmse:
        best_rmse = rmse
        best_params = params

print(f"\nBest parameters: {best_params}")
print(f"Best RMSE: {best_rmse:.4f}")

## 3. Optimization-Based Fitting

Optimization algorithms are more efficient. They start from an initial guess and iteratively improve the parameters to minimize prediction error.

In [ ]:
import pyactr as actr
from scipy.optimize import minimize, differential_evolution
import numpy as np

param_bounds = [
    (-2.0, 1.0),
    (0.01, 1.0),
    (0.1, 2.0),
]

def objective_function(params, human_data):
    """Objective function to minimize"""
    param_dict = {
        'retrieval_threshold': params[0],
        'latency_factor': params[1],
        'latency_exponent': params[2]
    }
    
    rmse = compute_rmse(param_dict, human_data)
    penalty = 0.01 * np.sum(np.abs(params))
    
    return rmse + penalty

print("1. Nelder-Mead Simplex Optimization")
initial_guess = [0.0, 0.2, 1.0]
result_nm = minimize(objective_function, initial_guess, 
                    args=(human_data,),
                    method='Nelder-Mead',
                    bounds=param_bounds)
print(f"   Best params: {result_nm.x}")
print(f"   Final error: {result_nm.fun:.4f}")

print("\n2. Differential Evolution (Global Optimization)")
result_de = differential_evolution(objective_function,
                                  param_bounds,
                                  args=(human_data,),
                                  seed=42,
                                  maxiter=50)
print(f"   Best params: {result_de.x}")
print(f"   Final error: {result_de.fun:.4f}")

best_params = dict(zip(['retrieval_threshold', 'latency_factor', 
                       'latency_exponent'], result_de.x))

print("\n3. Model Predictions vs Human Data:")
for size in human_data['set_size']:
    model_rt = simulate_response_time(best_params, size)
    human_rt = human_data['mean_rt'][human_data['set_size'].index(size)]
    print(f"   Set size {size}: Model={model_rt:.3f}s, Human={human_rt:.3f}s")

## 4. Cross-Validation for Model Assessment

A model that fits training data perfectly might not generalize to new data. Cross-validation helps detect overfitting by testing on held-out data.

In [ ]:
import pyactr as actr
import numpy as np
from sklearn.model_selection import KFold

np.random.seed(42)
trial_data = []

# Generate synthetic trial data
for set_size in [2, 4, 6, 8]:
    mean_rt = human_data['mean_rt'][human_data['set_size'].index(set_size)]
    std_rt = human_data['std_rt'][human_data['set_size'].index(set_size)]
    
    for _ in range(20):
        rt = np.random.normal(mean_rt, std_rt)
        trial_data.append({'set_size': set_size, 'rt': max(0.2, rt)})

trial_data = pd.DataFrame(trial_data)

# K-fold cross-validation
kfold = KFold(n_splits=5, shuffle=True, random_state=42)
cv_scores = []

for fold, (train_idx, test_idx) in enumerate(kfold.split(trial_data)):
    print(f"\nFold {fold + 1}:")
    
    train_data = trial_data.iloc[train_idx]
    test_data = trial_data.iloc[test_idx]
    
    train_summary = train_data.groupby('set_size')['rt'].agg(['mean', 'std'])
    human_train = {
        'set_size': list(train_summary.index),
        'mean_rt': list(train_summary['mean']),
        'std_rt': list(train_summary['std'])
    }
    
    result = minimize(objective_function, [0.0, 0.2, 1.0],
                     args=(human_train,),
                     method='Nelder-Mead')
    
    fitted_params = dict(zip(['retrieval_threshold', 'latency_factor',
                             'latency_exponent'], result.x))
    
    test_predictions = []
    test_actual = []
    
    for _, row in test_data.iterrows():
        pred_rt = simulate_response_time(fitted_params, row['set_size'])
        test_predictions.append(pred_rt)
        test_actual.append(row['rt'])
    
    test_rmse = np.sqrt(np.mean((np.array(test_predictions) - 
                                np.array(test_actual))**2))
    cv_scores.append(test_rmse)
    
    print(f"   Training RMSE: {result.fun:.4f}")
    print(f"   Test RMSE: {test_rmse:.4f}")

print(f"\nCross-validation Results:")
print(f"Mean CV RMSE: {np.mean(cv_scores):.4f} ± {np.std(cv_scores):.4f}")
print("\nGood generalization if test RMSE ≈ training RMSE!")

## 5. Bayesian Parameter Estimation

Instead of finding a single "best" parameter value, Bayesian methods estimate a distribution over plausible values. This gives you uncertainty estimates for free.

In [ ]:
import pyactr as actr
import numpy as np
import matplotlib.pyplot as plt

class BayesianFitter:
    def __init__(self, prior_means, prior_stds):
        self.prior_means = prior_means
        self.prior_stds = prior_stds
        self.samples = []
        
    def log_prior(self, params):
        """Log prior probability"""
        log_p = 0
        for i, (mean, std) in enumerate(zip(self.prior_means, self.prior_stds)):
            log_p += -0.5 * ((params[i] - mean) / std) ** 2
        return log_p
    
    def log_likelihood(self, params, data):
        """Log likelihood of data given parameters"""
        param_dict = {
            'retrieval_threshold': params[0],
            'latency_factor': params[1],
            'latency_exponent': params[2]
        }
        
        log_l = 0
        for i, size in enumerate(data['set_size']):
            predicted_rt = simulate_response_time(param_dict, size)
            observed_rt = data['mean_rt'][i]
            std_rt = data['std_rt'][i]
            
            log_l += -0.5 * ((predicted_rt - observed_rt) / std_rt) ** 2
        
        return log_l
    
    def sample_posterior(self, data, n_samples=1000):
        """Simple Metropolis-Hastings sampling"""
        current = np.array(self.prior_means)
        
        for i in range(n_samples):
            proposal = current + np.random.normal(0, 0.1, size=len(current))
            
            log_ratio = (self.log_prior(proposal) + 
                        self.log_likelihood(proposal, data) -
                        self.log_prior(current) - 
                        self.log_likelihood(current, data))
            
            if np.log(np.random.random()) < log_ratio:
                current = proposal
            
            if i > 200:
                self.samples.append(current.copy())
        
        return np.array(self.samples)

prior_means = [0.0, 0.2, 1.0]
prior_stds = [1.0, 0.2, 0.5]

fitter = BayesianFitter(prior_means, prior_stds)
posterior_samples = fitter.sample_posterior(human_data, n_samples=500)

param_names = ['Retrieval Threshold', 'Latency Factor', 'Latency Exponent']

print("Bayesian Parameter Estimates:")
for i, name in enumerate(param_names):
    samples = posterior_samples[:, i]
    print(f"\n{name}:")
    print(f"  Mean: {np.mean(samples):.3f}")
    print(f"  Std:  {np.std(samples):.3f}")
    print(f"  95% CI: [{np.percentile(samples, 2.5):.3f}, "
          f"{np.percentile(samples, 97.5):.3f}]")

fig, axes = plt.subplots(1, 3, figsize=(12, 4))
for i, (ax, name) in enumerate(zip(axes, param_names)):
    ax.hist(posterior_samples[:, i], bins=30, density=True, alpha=0.7)
    ax.axvline(prior_means[i], color='red', linestyle='--', label='Prior mean')
    ax.set_xlabel(name)
    ax.set_ylabel('Density')
    ax.legend()

plt.tight_layout()
plt.show()

## 6. Model Comparison and Selection

When you have competing theories, you need to compare models fairly. AIC and BIC penalize model complexity, preventing you from just adding more parameters.

In [ ]:
import pyactr as actr
import numpy as np

class LinearSearchModel:
    """Linear search through items"""
    def __init__(self, params):
        self.base_time = params[0]
        self.item_time = params[1]
        self.n_params = 2
    
    def predict_rt(self, set_size):
        return self.base_time + self.item_time * set_size

class LogSearchModel:
    """Logarithmic search (more efficient)"""
    def __init__(self, params):
        self.base_time = params[0]
        self.log_factor = params[1]
        self.power = params[2]
        self.n_params = 3
    
    def predict_rt(self, set_size):
        return self.base_time + self.log_factor * (set_size ** self.power)

def fit_model(model_class, data):
    """Fit a model and return best parameters and fit statistics"""
    
    def objective(params):
        model = model_class(params)
        predicted = [model.predict_rt(size) for size in data['set_size']]
        return np.sum((np.array(predicted) - np.array(data['mean_rt']))**2)
    
    if model_class == LinearSearchModel:
        bounds = [(0.1, 1.0), (0.01, 0.2)]
        x0 = [0.3, 0.1]
    else:
        bounds = [(0.1, 1.0), (0.01, 1.0), (0.1, 2.0)]
        x0 = [0.3, 0.2, 0.5]
    
    result = minimize(objective, x0, bounds=bounds)
    
    model = model_class(result.x)
    predicted = [model.predict_rt(size) for size in data['set_size']]
    residuals = np.array(data['mean_rt']) - np.array(predicted)
    
    sse = np.sum(residuals**2)
    n = len(data['set_size'])
    k = model.n_params
    
    aic = n * np.log(sse/n) + 2 * k
    bic = n * np.log(sse/n) + k * np.log(n)
    
    return {
        'params': result.x,
        'sse': sse,
        'aic': aic,
        'bic': bic,
        'model': model
    }

linear_fit = fit_model(LinearSearchModel, human_data)
log_fit = fit_model(LogSearchModel, human_data)

print("Model Comparison Results:")
print("\n1. Linear Search Model:")
print(f"   Parameters: {linear_fit['params']}")
print(f"   SSE: {linear_fit['sse']:.4f}")
print(f"   AIC: {linear_fit['aic']:.2f}")
print(f"   BIC: {linear_fit['bic']:.2f}")

print("\n2. Logarithmic Search Model:")
print(f"   Parameters: {log_fit['params']}")
print(f"   SSE: {log_fit['sse']:.4f}")
print(f"   AIC: {log_fit['aic']:.2f}")
print(f"   BIC: {log_fit['bic']:.2f}")

print("\n3. Model Selection:")
aic_diff = linear_fit['aic'] - log_fit['aic']
bic_diff = linear_fit['bic'] - log_fit['bic']

print(f"   ΔAIC (Linear - Log): {aic_diff:.2f}")
print(f"   ΔBIC (Linear - Log): {bic_diff:.2f}")

if aic_diff > 2:
    print("   → Logarithmic model strongly preferred (lower AIC)")
elif aic_diff < -2:
    print("   → Linear model strongly preferred (lower AIC)")
else:
    print("   → Models roughly equivalent")

set_sizes = np.linspace(1, 10, 100)
linear_pred = [linear_fit['model'].predict_rt(s) for s in set_sizes]
log_pred = [log_fit['model'].predict_rt(s) for s in set_sizes]

plt.figure(figsize=(8, 6))
plt.scatter(human_data['set_size'], human_data['mean_rt'], 
           s=100, label='Human Data', color='black')
plt.plot(set_sizes, linear_pred, label='Linear Model', color='blue')
plt.plot(set_sizes, log_pred, label='Log Model', color='red')
plt.xlabel('Set Size')
plt.ylabel('Response Time (s)')
plt.legend()
plt.grid(True, alpha=0.3)
plt.show()

## 7. Complete Example: Fitting Memory Model to Data

Here's a fuller example using recognition memory data with multiple dependent variables (accuracy and RT).

In [ ]:
import pyactr as actr
import numpy as np
import matplotlib.pyplot as plt

recognition_data = {
    'delay': [0, 1, 5, 10, 30, 60],
    'p_correct': [0.95, 0.88, 0.75, 0.65, 0.45, 0.35],
    'mean_rt': [0.8, 0.95, 1.15, 1.35, 1.6, 1.85]
}

def create_memory_model(decay_rate, threshold, noise):
    """Create ACT-R model with specified memory parameters"""
    model = actr.ACTRModel()
    model.model_parameters['subsymbolic'] = True
    model.model_parameters['decay'] = decay_rate
    model.model_parameters['retrieval_threshold'] = threshold
    model.model_parameters['instantaneous_noise'] = noise
    model.model_parameters['latency_factor'] = 0.4
    model.model_parameters['latency_exponent'] = 1.0
    
    actr.chunktype("memory_item", "word study_time")
    actr.chunktype("recognition_goal", "state word response")
    
    return model

def simulate_recognition(model, delay, n_trials=100):
    """Simulate recognition at given delay"""
    decay = model.model_parameters['decay']
    threshold = model.model_parameters['retrieval_threshold']
    noise = model.model_parameters['instantaneous_noise']
    
    successes = 0
    rts = []
    
    for _ in range(n_trials):
        if delay == 0:
            base_activation = 0
        else:
            base_activation = -decay * np.log(delay + 1)
        
        activation = base_activation + np.random.normal(0, noise)
        
        if activation > threshold:
            successes += 1
            rt = 0.4 * np.exp(-activation)
            rts.append(rt)
    
    p_correct = successes / n_trials
    mean_rt = np.mean(rts) if rts else 2.0
    
    return p_correct, mean_rt

def fit_memory_model(data):
    """Fit model to recognition data"""
    
    def objective(params):
        decay, threshold, noise = params
        model = create_memory_model(decay, threshold, noise)
        
        error = 0
        for i, delay in enumerate(data['delay']):
            p_pred, rt_pred = simulate_recognition(model, delay)
            
            error += (p_pred - data['p_correct'][i])**2
            error += 0.1 * (rt_pred - data['mean_rt'][i])**2
        
        return error
    
    bounds = [(0.1, 1.0), (-2.0, 0.0), (0.1, 0.5)]
    x0 = [0.5, -0.5, 0.25]
    
    result = minimize(objective, x0, bounds=bounds, method='L-BFGS-B')
    return result.x

print("Fitting memory model to human data...")
best_params = fit_memory_model(recognition_data)
decay, threshold, noise = best_params

print(f"\nBest fit parameters:")
print(f"  Decay rate: {decay:.3f}")
print(f"  Threshold: {threshold:.3f}")
print(f"  Noise: {noise:.3f}")

model = create_memory_model(decay, threshold, noise)
model_p_correct = []
model_rt = []

for delay in recognition_data['delay']:
    p, rt = simulate_recognition(model, delay)
    model_p_correct.append(p)
    model_rt.append(rt)

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(12, 5))

ax1.scatter(recognition_data['delay'], recognition_data['p_correct'], 
           s=100, label='Human', color='black')
ax1.plot(recognition_data['delay'], model_p_correct, 
        'r-', label='Model', linewidth=2)
ax1.set_xlabel('Delay (minutes)')
ax1.set_ylabel('P(Correct)')
ax1.set_title('Recognition Accuracy')
ax1.legend()
ax1.grid(True, alpha=0.3)

ax2.scatter(recognition_data['delay'], recognition_data['mean_rt'], 
           s=100, label='Human', color='black')
ax2.plot(recognition_data['delay'], model_rt, 
        'r-', label='Model', linewidth=2)
ax2.set_xlabel('Delay (minutes)')
ax2.set_ylabel('Response Time (s)')
ax2.set_title('Recognition RT')
ax2.legend()
ax2.grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

r2_accuracy = 1 - np.var(np.array(model_p_correct) - 
                        np.array(recognition_data['p_correct'])) / \
                  np.var(recognition_data['p_correct'])
print(f"\nModel fit quality:")
print(f"  R² for accuracy: {r2_accuracy:.3f}")

## Exercises

1. Modify the fitting procedure to simultaneously fit both accuracy and response time data with different weighting schemes.

2. Extend the approach to handle multiple participants, each with their own parameters.

3. Implement a model that updates its parameters as new data arrives.

4. Test whether your fitting procedure can recover known parameters from simulated data (parameter recovery analysis).

5. Implement a hierarchical Bayesian model that estimates both group-level and individual parameters.

## Summary

This tutorial covered the main approaches to fitting ACT-R models to data. Grid search works but is slow for high-dimensional spaces. Optimization algorithms are faster and often find good solutions. Bayesian methods give you uncertainty estimates, which is critical when reporting results. Cross-validation helps you avoid overfitting, and information criteria (AIC/BIC) let you compare models with different numbers of parameters.

Next steps: look into hierarchical models if you have data from multiple participants, or sequential parameter estimation if your data arrives over time.